In [1]:
%load_ext autoreload
%autoreload 2

# Plotly Global Instance Explorer

`plotly.global.instance_explorer` is invoked through CE's global plotting API: `explainer.plot(X_test)` or `explainer.plot(X_test, y_test)`. It is a hover-only batch overview in prediction/uncertainty space, not a global CE explanation method. Marker size indicates how many instances share a deterministic aggregated plotted position.

In [2]:
import ce_visualization_plotly.plugin as plotly_plugin
import numpy as np
from calibrated_explanations import WrapCalibratedExplainer
from crepes.extras import DifficultyEstimator
from sklearn.datasets import make_classification, make_regression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split

plotly_plugin.register_plotly_visualization_components()  # explicit, idempotent registration


## Section 1: Classification

The classification explorer is called through the normal global plotting API. Without targets, the plot follows the predicted-class probability convention. With targets supplied, CE global probabilistic classification plots use target-aware marker symbols. The Plotly explorer follows that convention and keeps the probability triangle reference shape.

In [3]:
X, y = make_classification(
    n_samples=500,
    n_features=8,
    n_informative=5,
    n_redundant=0,
    random_state=7,
)
x_proper, x_tmp, y_proper, y_tmp = train_test_split(
    X, y, test_size=0.4, random_state=7, stratify=y
)
x_cal, x_query, y_cal, y_query = train_test_split(
    x_tmp, y_tmp, test_size=0.5, random_state=7, stratify=y_tmp
)

classifier = RandomForestClassifier(n_estimators=80, random_state=7)
explainer = WrapCalibratedExplainer(classifier)
explainer.fit(x_proper, y_proper)
assert explainer.fitted is True
explainer.calibrate(x_cal, y_cal)
assert explainer.calibrated is True

classification_result = explainer.plot(
    x_query[:120],
    style="plotly.global.instance_explorer",
    task="classification",
    position_precision=2,
    show=True,
)

classification_result_with_targets = explainer.plot(
    x_query[:120],
    y_query[:120],
    style="plotly.global.instance_explorer",
    task="classification",
    position_precision=2,
    show=True,
)

## Section 2: Probabilistic Regression / Thresholded Regression

For thresholded regression, the x-axis is the calibrated probability of the target event. Targets are converted into event classes by the global plotting payload, and hover reports the threshold event and calibrated probability interval.

In [4]:
X_reg2, y_reg2 = make_regression(
    n_samples=500,
    n_features=7,
    n_informative=5,
    noise=10.0,
    random_state=19,
)
x_proper, x_tmp, y_proper, y_tmp = train_test_split(
    X_reg2, y_reg2, test_size=0.4, random_state=19
)
x_cal, x_query, y_cal, y_query = train_test_split(
    x_tmp, y_tmp, test_size=0.5, random_state=19
)

interval_regressor = RandomForestRegressor(n_estimators=80, random_state=19)
interval_explainer = WrapCalibratedExplainer(interval_regressor)
interval_explainer.fit(x_proper, y_proper)
assert interval_explainer.fitted is True
interval_explainer.calibrate(x_cal, y_cal)
assert interval_explainer.calibrated is True

threshold = float(np.median(y_proper))
threshold_result = interval_explainer.plot(
    x_query[:120],
    y_query[:120],
    threshold=threshold,
    style="plotly.global.instance_explorer",
    task="probabilistic_regression",
    position_precision=2,
    show=True,
)

interval_explainer.set_difficulty_estimator(
    DifficultyEstimator().fit(X=x_proper, learner=interval_explainer.learner, scaler=True)
)

threshold = float(np.median(y_proper))
threshold_result = interval_explainer.plot(
    x_query[:120],
    y_query[:120],
    threshold=threshold,
    style="plotly.global.instance_explorer",
    task="probabilistic_regression",
    position_precision=2,
    show=True,
)

## Section 3: Non-Probabilistic Regression

Without a threshold, CE global regression plots use predictions on the x-axis and uncertainty on the y-axis. When targets are supplied, the target values are represented by marker color.

In [5]:
regression_result = interval_explainer.plot(
    x_query[:120],
    y_query[:120],
    style="plotly.global.instance_explorer",
    task="regression",
    position_precision=2,
    show=True,
)